# Module 4 - Class 6 LAB: GridSearch and Model Selection

**Dataset:** Telco Customer Churn  
**Objective:** Compare multiple models, tune hyperparameters, and select the best model.

### What you will do
- Train Logistic Regression, Random Forest, and SVM
- Tune Random Forest with GridSearchCV
- Compare all models on multiple metrics
- Plot combined ROC curves
- Write a model selection report

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, classification_report
)
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")

## 1. Load and Preprocess Data

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

replace_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in replace_cols:
    df[col] = df[col].replace('No internet service', 'No')
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

df['Churn'] = (df['Churn'] == 'Yes').astype(int)

X = df.drop(['customerID', 'Churn'], axis=1)
y = df['Churn']

numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

# Preprocessor
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, cat_cols)
])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Churn rate - Train: {y_train.mean():.2%}, Test: {y_test.mean():.2%}")

## 2. Train Baseline Models

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42)
}

# Train each model in a pipeline
results = {}
trained_pipes = {}

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        'y_pred': y_pred,
        'y_proba': y_proba
    }
    trained_pipes[name] = pipe
    print(f"{name} trained.")

print("\nAll models trained.")

## 3. GridSearchCV on Random Forest

In [ ]:
# Define parameter grid
param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [5, 10, 15, None]
}

# Pipeline for grid search
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    rf_pipe,
    param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV F1 score: {grid_search.best_score_:.4f}")

In [ ]:
# Evaluate best RF on test set
best_rf = grid_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
y_proba_best = best_rf.predict_proba(X_test)[:, 1]

results['RF (Tuned)'] = {
    'Accuracy': accuracy_score(y_test, y_pred_best),
    'Precision': precision_score(y_test, y_pred_best),
    'Recall': recall_score(y_test, y_pred_best),
    'F1': f1_score(y_test, y_pred_best),
    'ROC-AUC': roc_auc_score(y_test, y_proba_best),
    'y_pred': y_pred_best,
    'y_proba': y_proba_best
}
trained_pipes['RF (Tuned)'] = best_rf

print("Tuned RF Results:")
print(classification_report(y_test, y_pred_best, target_names=['No Churn', 'Churn']))

In [ ]:
# GridSearch results heatmap
cv_results = pd.DataFrame(grid_search.cv_results_)
pivot = cv_results.pivot_table(
    values='mean_test_score',
    index='param_classifier__max_depth',
    columns='param_classifier__n_estimators'
)

plt.figure(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd')
plt.title('GridSearchCV Mean F1 Scores')
plt.xlabel('n_estimators')
plt.ylabel('max_depth')
plt.show()

## 4. Model Comparison Table

In [ ]:
comparison = pd.DataFrame([
    {'Model': name, **{k: v for k, v in metrics.items() if k not in ['y_pred', 'y_proba']}}
    for name, metrics in results.items()
]).round(4)

print(comparison.to_string(index=False))

## 5. Combined ROC Curve

In [ ]:
plt.figure(figsize=(8, 7))

colors = ['#1976d2', '#388e3c', '#d32f2f', '#f57c00']
for i, (name, metrics) in enumerate(results.items()):
    fpr, tpr, _ = roc_curve(y_test, metrics['y_proba'])
    auc_val = metrics['ROC-AUC']
    plt.plot(fpr, tpr, color=colors[i], linewidth=2,
             label=f"{name} (AUC = {auc_val:.3f})")

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - All Models', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. MLflow Logging (Optional Demo)

MLflow tracks experiments: parameters, metrics, and models. This is optional -- uncomment and run if you want to try it.

In [ ]:
# Uncomment to install and use MLflow
# !pip install mlflow -q

# import mlflow
# 
# mlflow.set_experiment("telco_churn_model_selection")
# 
# for name, metrics in results.items():
#     with mlflow.start_run(run_name=name):
#         mlflow.log_param("model_type", name)
#         mlflow.log_metric("accuracy", metrics['Accuracy'])
#         mlflow.log_metric("precision", metrics['Precision'])
#         mlflow.log_metric("recall", metrics['Recall'])
#         mlflow.log_metric("f1", metrics['F1'])
#         mlflow.log_metric("roc_auc", metrics['ROC-AUC'])
#         print(f"Logged: {name}")
# 
# print("\nAll runs logged. View with: mlflow ui")

## 7. TODO: Model Selection Report

Fill in each section below with your analysis. Use the results from the experiments above.

### 7.1 Problem Statement

**TODO:** In 2-3 sentences, describe the problem. What are we predicting? Why does it matter?

*Your answer here:*



### 7.2 Models Evaluated

**TODO:** List the models you compared and briefly describe each (1 sentence each).

| Model | Description |
|-------|-------------|
| Logistic Regression | |
| Random Forest | |
| SVM | |
| RF (Tuned) | |

### 7.3 Results Analysis

**TODO:** Answer these questions (2-3 sentences each):

1. Which model had the highest accuracy? Is accuracy the right metric here? Why or why not?

*Your answer:*

2. Which model had the best recall? Why is recall important for churn prediction?

*Your answer:*

3. Did GridSearchCV improve the Random Forest? By how much?

*Your answer:*


### 7.4 Final Recommendation

**TODO:** Which model would you deploy for churn prediction? Justify your choice considering:
- Performance metrics
- Business context (cost of missing a churner vs false alarms)
- Interpretability needs

*Your recommendation (3-5 sentences):*



### 7.5 Next Steps

**TODO:** What would you try next to improve the model? List 2-3 ideas.

1. 
2. 
3. 

---
## Summary

| Concept | Details |
|---------|---------|
| GridSearchCV | Exhaustive search over parameter combinations with cross-validation |
| StratifiedKFold | Preserves class ratio in each fold |
| Model comparison | Evaluate multiple metrics, not just accuracy |
| ROC curves | Visual comparison of model discrimination ability |
| MLflow | Experiment tracking: log params, metrics, models |